# 1. Application Security Design

SC-100 asks: **"Design a secure application lifecycle — from threat modeling through DevSecOps to production workload protection."**

## Setup

```bash
cd security-certs/sc-100/04-applications-and-data
uv sync
# Notebooks use the local .venv directly -- no global kernel to register.
# In VS Code: open the kernel picker (top-right) and select `.venv`.
# In classic Jupyter: uv run jupyter notebook notebooks/
```
Then pick the **`.venv` kernel** for this folder from the VS Code kernel picker (top-right).

If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window").

## Threat Modeling with STRIDE

Threat modeling is **the most important security design activity**. It should happen early in the design phase, not after deployment.

### STRIDE categories:

| Letter | Threat | Property violated | Example |
|--------|--------|------------------|---------|
| **S** | Spoofing | Authentication | Attacker impersonates a user or service |
| **T** | Tampering | Integrity | Attacker modifies data in transit or at rest |
| **R** | Repudiation | Non-repudiation | User denies performing an action, no audit trail |
| **I** | Information Disclosure | Confidentiality | Sensitive data exposed (PII, secrets, tokens) |
| **D** | Denial of Service | Availability | Application becomes unavailable |
| **E** | Elevation of Privilege | Authorization | User gains unauthorized access to higher privilege |

### Threat modeling process:

```
1. DEFINE SCOPE
   └── What are we modeling? (system, feature, data flow)
         │
2. CREATE DATA FLOW DIAGRAM (DFD)
   └── Identify: processes, data stores, external entities, data flows, trust boundaries
         │
3. IDENTIFY THREATS (STRIDE per element)
   └── For each element in the DFD, ask: what could go wrong?
         │
4. RATE AND PRIORITIZE (DREAD or risk matrix)
   └── Likelihood × Impact = Risk priority
         │
5. DETERMINE MITIGATIONS
   └── Design controls: what Azure/Microsoft features address each threat?
         │
6. VALIDATE
   └── Review with dev team, update as architecture evolves
```

In [ ]:
import json

# ===================================================================
# THREAT MODELING EXERCISE
# Apply STRIDE to a real-world e-commerce application
# ===================================================================

APP_DESCRIPTION = """
APPLICATION: Tailspin Toys Online Store

Architecture:
  User (browser) → Azure Front Door → AKS (API) → Azure SQL (data)
                                         ↓
                                    Key Vault (secrets)
                                         ↓
                                   Storage (product images)
                                         ↓
                                  Payment Gateway (3rd party)

Trust boundaries:
  TB1: Internet → Azure Front Door (public to private)
  TB2: AKS → Azure SQL (app to data)
  TB3: AKS → Payment Gateway (internal to external 3rd party)
"""

STRIDE_ANALYSIS = [
    {
        'element': 'User → Front Door (TB1)',
        'threat': 'Spoofing',
        'description': 'Attacker creates fake login page to steal credentials',
        'risk': 'High',
        'mitigation': 'Entra ID authentication (federated, not custom login form). FIDO2/passkeys for phishing resistance.',
        'azure_feature': 'Entra External ID (CIAM) with phishing-resistant MFA',
    },
    {
        'element': 'User → Front Door (TB1)',
        'threat': 'Tampering',
        'description': 'Attacker manipulates HTTP requests (e.g., changes price in cart)',
        'risk': 'High',
        'mitigation': 'Server-side validation of all inputs. WAF on Front Door. Input sanitization.',
        'azure_feature': 'Azure WAF managed rules (CRS 3.2 on App Gateway / DRS 2.1 on Front Door) + server-side input validation',
    },
    {
        'element': 'AKS API',
        'threat': 'Repudiation',
        'description': 'Customer disputes a purchase, no proof of transaction',
        'risk': 'Medium',
        'mitigation': 'Comprehensive audit logging. Immutable transaction log. Digital receipt.',
        'azure_feature': 'Application Insights + Sentinel for transaction audit trail',
    },
    {
        'element': 'AKS → Azure SQL (TB2)',
        'threat': 'Information Disclosure',
        'description': 'SQL injection exposes customer PII and payment data',
        'risk': 'Critical',
        'mitigation': 'Parameterized queries (ORM). Encrypt sensitive columns. Private endpoint.',
        'azure_feature': 'Azure SQL Always Encrypted, Defender for SQL threat detection, Private Endpoint',
    },
    {
        'element': 'Front Door',
        'threat': 'Denial of Service',
        'description': 'DDoS attack during Black Friday sale takes store offline',
        'risk': 'High',
        'mitigation': 'DDoS protection, rate limiting, CDN caching, auto-scaling AKS.',
        'azure_feature': 'Azure DDoS Network Protection (L3/L4) + Front Door WAF rate limiting (L7) + AKS HPA',
    },
    {
        'element': 'AKS → Payment Gateway (TB3)',
        'threat': 'Elevation of Privilege',
        'description': 'Compromised container escalates to access payment gateway credentials',
        'risk': 'Critical',
        'mitigation': 'Workload identity (no stored credentials). Pod security standards. Network policies.',
        'azure_feature': 'AKS Workload Identity + Key Vault CSI + Pod Security Admission (restricted)',
    },
    {
        'element': 'Key Vault',
        'threat': 'Information Disclosure',
        'description': 'Attacker accesses stored secrets (DB connection strings, API keys)',
        'risk': 'Critical',
        'mitigation': 'Private endpoint. RBAC (not access policies). Managed identity access only. Audit logging.',
        'azure_feature': 'Key Vault with Private Endpoint, RBAC, Defender for Key Vault, diagnostic logs → Sentinel',
    },
]

print(APP_DESCRIPTION)
print('=== STRIDE Threat Analysis ===\n')
print(f'{"Element":<30} {"STRIDE":<12} {"Risk":<10} {"Azure Mitigation"}')
print('─' * 120)
for t in STRIDE_ANALYSIS:
    print(f'{t["element"]:<30} {t["threat"]:<12} {t["risk"]:<10} {t["azure_feature"]}')

## DevSecOps Pipeline Design

### Shift-left security: embed security at every stage of the pipeline

```
CODE          BUILD           TEST           DEPLOY          OPERATE
 │              │               │               │               │
 ▼              ▼               ▼               ▼               ▼
┌──────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│Pre-   │    │ SAST     │    │ DAST     │    │ Infra    │    │ Runtime  │
│commit │    │ SCA      │    │ Pen test │    │ scanning │    │ protect  │
│hooks  │    │ Container│    │ Fuzz     │    │ Policy   │    │ Monitor  │
│       │    │ scan     │    │ testing  │    │ gates    │    │ Respond  │
└──────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
 • Secrets   • Dependency    • API security   • IaC scanning  • Defender
   scanning    vulnerabilities • Auth testing    (Bicep/TF)   • Sentinel
 • Linting   • Code quality  • Injection       • Azure Policy • Incident
 • .gitignore• License check    testing          compliance     response
 • Credential• Image CVEs   • Business logic  • Approval      • Patching
   detection                    testing          gates
```

### Microsoft tools for each stage:

| Stage | Tool | Purpose |
|-------|------|---------|
| Code | GitHub Advanced Security (GHAS) | Secret scanning, code scanning (CodeQL) |
| Code | Credential Scanner (Azure DevOps) | Detect secrets in code |
| Build | Defender for Cloud **DevOps security** (the GA name for what preview called "Defender for DevOps") | Connect GitHub / Azure DevOps / GitLab to Defender for Cloud; surfaces code, secret and IaC findings next to your cloud posture |
| Build | Defender for Containers | Image vulnerability scanning in ACR |
| Build | GitHub Dependabot / ADO Component Governance | SCA (dependency scanning) |
| Deploy | Azure Policy | Prevent non-compliant deployments |
| Deploy | Deployment gates (ADO) | Require security approval for prod |
| Operate | Defender for Cloud | Runtime protection |
| Operate | Microsoft Sentinel | Threat detection and response |

In [ ]:
# ===================================================================
# DEVSECOPS PIPELINE DESIGN EXERCISE
# Design security gates for a CI/CD pipeline
# ===================================================================

PIPELINE_SECURITY_GATES = [
    {
        'stage': 'Pre-commit',
        'gate': 'Secret detection',
        'tool': 'GitHub secret scanning push protection',
        'action': 'Block commit if secrets detected',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'Static Application Security Testing (SAST)',
        'tool': 'CodeQL (via GitHub Advanced Security)',
        'action': 'Flag high/critical findings, require fix before merge',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'Software Composition Analysis (SCA)',
        'tool': 'Dependabot / GitHub Dependency Review',
        'action': 'Block PR if new dependency has critical CVE',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'Container image scan',
        'tool': 'Defender for Containers (ACR task)',
        'action': 'Scan image on push to ACR, quarantine if critical CVEs',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'IaC scanning',
        'tool': 'Defender for Cloud DevOps security (Bicep/Terraform/ARM scanning)',
        'action': 'Detect misconfigurations in infrastructure code',
        'blocking': False,
    },
    {
        'stage': 'Staging',
        'gate': 'DAST (Dynamic Application Security Testing)',
        'tool': 'OWASP ZAP or commercial DAST tool',
        'action': 'Run against staging environment, report findings',
        'blocking': False,
    },
    {
        'stage': 'Pre-production',
        'gate': 'Security approval gate',
        'tool': 'Azure DevOps / GitHub environment protection rules',
        'action': 'Security team must approve production deployments',
        'blocking': True,
    },
    {
        'stage': 'Production deploy',
        'gate': 'Azure Policy compliance check',
        'tool': 'Azure Policy (deny non-compliant resources)',
        'action': 'Deployment fails if resources violate policy',
        'blocking': True,
    },
]

print('=== DevSecOps Security Gates ===\n')
print(f'{"Stage":<18} {"Gate":<40} {"Tool":<40} {"Blocking?"}')
print('─' * 120)
for gate in PIPELINE_SECURITY_GATES:
    blocking = 'BLOCKING' if gate['blocking'] else 'Advisory'
    print(f'{gate["stage"]:<18} {gate["gate"]:<40} {gate["tool"]:<40} {blocking}')

blocking_count = sum(1 for g in PIPELINE_SECURITY_GATES if g['blocking'])
print(f'\nBlocking gates: {blocking_count}/{len(PIPELINE_SECURITY_GATES)}')
print('Design principle: Block on high-severity issues only. Advisory for medium/low to avoid developer friction.')

In [ ]:
# ===================================================================
# WORKLOAD IDENTITY DESIGN
# How applications authenticate to Azure services
# ===================================================================

WORKLOAD_IDENTITY_OPTIONS = [
    {
        'scenario': 'App Service accessing Azure SQL',
        'recommended': 'System-assigned managed identity',
        'why': 'Simplest, lifecycle tied to the app. No credential management.',
        'avoid': 'Connection string with SQL username/password',
    },
    {
        'scenario': 'Multiple apps accessing same Key Vault',
        'recommended': 'User-assigned managed identity (shared)',
        'why': 'One identity, one RBAC assignment. Survives app redeployment.',
        'avoid': 'Separate service principals with client secrets',
    },
    {
        'scenario': 'GitHub Actions deploying to Azure',
        'recommended': 'Workload identity federation (OIDC)',
        'why': 'No secrets stored in GitHub. GitHub token exchanged for Azure token.',
        'avoid': 'Service principal with client secret stored in GitHub Secrets',
    },
    {
        'scenario': 'AKS pods accessing Azure services',
        'recommended': 'AKS Workload Identity (federated)',
        'why': 'Pod-level identity. No shared node identity. Kubernetes SA projected token.',
        'avoid': 'Pod Identity (v1, deprecated) or shared AKS kubelet identity',
    },
    {
        'scenario': 'On-premises app accessing Azure',
        'recommended': 'Managed identity via Azure Arc (if possible) or certificate-based SP',
        'why': 'Arc-enabled servers get managed identities. Otherwise, certificate > secret.',
        'avoid': 'Service principal with long-lived client secret',
    },
    {
        'scenario': 'Multi-tenant SaaS app accessing customer tenants',
        'recommended': 'Multi-tenant app registration with admin consent',
        'why': 'Single app registration in your tenant, customers consent to access.',
        'avoid': 'Separate app registrations per customer',
    },
]

print('=== Workload Identity Design Guide ===\n')
for wi in WORKLOAD_IDENTITY_OPTIONS:
    print(f'\n--- {wi["scenario"]} ---')
    print(f'  Recommended: {wi["recommended"]}')
    print(f'  Why: {wi["why"]}')
    print(f'  Avoid: {wi["avoid"]}')

print('\n\nIdentity preference order (most to least secure):')
print('  1. Managed identity (system-assigned)')
print('  2. Managed identity (user-assigned)')
print('  3. Workload identity federation (OIDC)')
print('  4. Service principal with certificate')
print('  5. Service principal with client secret (LAST RESORT)')

In [ ]:
# ===================================================================
# API SECURITY ARCHITECTURE
# ===================================================================

API_SECURITY_DESIGN = {
    'Azure API Management (APIM)': {
        'role': 'API gateway — centralized security, rate limiting, authentication',
        'security_features': [
            'OAuth 2.0 / OpenID Connect validation (Entra ID)',
            'Subscription keys for API consumers',
            'Rate limiting and throttling policies',
            'IP filtering and geo-restriction',
            'Request/response transformation (strip sensitive headers)',
            'Certificate validation for mTLS',
            'WAF integration via Application Gateway',
        ],
    },
    'API authentication patterns': {
        'internal_apis': 'Managed identity (service-to-service within Azure)',
        'partner_apis': 'OAuth 2.0 client credentials flow with certificate',
        'customer_apis': 'OAuth 2.0 authorization code flow with PKCE',
        'webhook_apis': 'HMAC signature validation + IP allowlist',
    },
    'WAF deployment patterns': {
        'Global apps (multi-region)': 'Azure Front Door + WAF',
        'Regional apps + APIM': 'Application Gateway + WAF → APIM → Backend',
        'Simple web apps': 'Application Gateway + WAF → App Service',
        'APIs only': 'APIM policies (rate limit, IP filter, JWT validation)',
    },
}

print('=== API Security Architecture ===\n')
for category, details in API_SECURITY_DESIGN.items():
    print(f'\n{"=" * 70}')
    print(f'{category}')
    print(f'{"=" * 70}')
    if isinstance(details, dict):
        if 'role' in details:
            print(f'  Role: {details["role"]}')
            for feature in details['security_features']:
                print(f'  • {feature}')
        else:
            for key, value in details.items():
                print(f'  {key}: {value}')

## Secure Code Patterns — Bad vs Best (Runnable)

SC-100 is a design exam, but good design shows up in code. The following cells demonstrate **bad → best** patterns that correspond to the STRIDE threats above. Every cell uses only the Python standard library so you can run it anywhere.

| STRIDE | Bad pattern | Best pattern | Demo |
|--------|-------------|--------------|------|
| Tampering / Info Disclosure | String-concatenated SQL | Parameterized query | `sqlite3` |
| Spoofing (webhooks) | Trust the sender | HMAC signature verification | `hmac` |
| Info Disclosure (passwords) | MD5/SHA-1 of password | `scrypt` with random salt | `hashlib` |
| Tampering (input) | Trust client input | Strict server-side validation | `dataclasses` + checks |
| Info Disclosure (secrets) | Hardcoded credentials | Fetch from env / Key Vault | `os.environ` |


In [ ]:
# -------------------------------------------------------------------
# 1) SQL Injection — Tampering + Information Disclosure
# -------------------------------------------------------------------
# Attack scenario: customer support page looks up orders by email.
# An attacker supplies:   x' OR '1'='1
# If you build the query by string concatenation, the attacker dumps the whole table.
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
    CREATE TABLE orders (id INTEGER, email TEXT, total REAL);
    INSERT INTO orders VALUES (1, 'alice@example.com', 42.00);
    INSERT INTO orders VALUES (2, 'bob@example.com',  99.00);
    INSERT INTO orders VALUES (3, 'carol@example.com', 12.50);
""")

attacker_input = "x' OR '1'='1"

# ❌ BAD — string concatenation, vulnerable to SQL injection
bad_query = f"SELECT id, email, total FROM orders WHERE email = '{attacker_input}'"
print("BAD query :", bad_query)
print("BAD result:", db.execute(bad_query).fetchall())

# ✅ BEST — parameterized query, driver escapes the value
best_rows = db.execute(
    "SELECT id, email, total FROM orders WHERE email = ?",
    (attacker_input,),
).fetchall()
print("BEST result:", best_rows, "  ← no rows leaked\n")

print("Azure mapping: Defender for SQL detects SQL injection at runtime,")
print("but prevention belongs in code (ORM / parameterized queries).")


In [ ]:
# -------------------------------------------------------------------
# 2) Webhook signature verification — Spoofing
# -------------------------------------------------------------------
# A 3rd party (payment gateway) POSTs to your /webhook endpoint.
# Without verification, anyone on the internet can forge "payment succeeded" events.
import hmac, hashlib, secrets

SHARED_SECRET = secrets.token_bytes(32)  # stored in Key Vault in production

def sign(body: bytes, secret: bytes) -> str:
    return hmac.new(secret, body, hashlib.sha256).hexdigest()

def verify(body: bytes, signature: str, secret: bytes) -> bool:
    expected = sign(body, secret)
    return hmac.compare_digest(expected, signature)  # constant-time compare

legit_body = b'{"order_id":123,"status":"paid"}'
legit_sig  = sign(legit_body, SHARED_SECRET)

forged_body = b'{"order_id":123,"status":"paid","amount":0}'
forged_sig  = "deadbeef" * 8  # attacker guess

# ❌ BAD — trust the payload because it arrived at your URL
print("BAD  (no check): accepted forged request")

# ✅ BEST — verify signature before processing
print("BEST legit  ->", verify(legit_body,  legit_sig,  SHARED_SECRET))
print("BEST forged ->", verify(forged_body, forged_sig, SHARED_SECRET))

print("\nAzure mapping: APIM 'validate-jwt' for OAuth-based partners;")
print("for HMAC webhooks, validate in an APIM policy or in the API itself.")


In [ ]:
# -------------------------------------------------------------------
# 3) Password storage — Information Disclosure on breach
# -------------------------------------------------------------------
# If your DB is leaked, weak hashing = instant credential stuffing across the internet.
import hashlib, secrets

password = "correct-horse-battery-staple"

# ❌ BAD — fast unsalted hash (crackable at billions/sec on a GPU)
bad_hash = hashlib.md5(password.encode()).hexdigest()
print("BAD  md5  :", bad_hash)

# ❌ Still bad — SHA-256 is also fast and unsalted
bad_hash2 = hashlib.sha256(password.encode()).hexdigest()
print("BAD  sha256:", bad_hash2)

# ✅ BEST — memory-hard KDF (scrypt) with a unique random salt per user
salt = secrets.token_bytes(16)
good_hash = hashlib.scrypt(password.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
stored = salt.hex() + ":" + good_hash.hex()  # store this in the DB
print("BEST scrypt:", stored[:80], "...")

# Verification (constant-time)
def verify_password(candidate: str, stored: str) -> bool:
    salt_hex, hash_hex = stored.split(":")
    salt = bytes.fromhex(salt_hex)
    expected = bytes.fromhex(hash_hex)
    actual = hashlib.scrypt(candidate.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
    return secrets.compare_digest(expected, actual)

print("verify correct :", verify_password(password, stored))
print("verify wrong   :", verify_password("guess", stored))

print("\nBest for production: Argon2id (winner of the Password Hashing Competition).")
print("Better still: skip passwords — use Entra ID + FIDO2/passkeys (phishing-resistant).")


In [ ]:
# -------------------------------------------------------------------
# 4) Input validation — Tampering (price manipulation from the threat model)
# -------------------------------------------------------------------
from dataclasses import dataclass

PRICE_BOOK = {"SKU-001": 9.99, "SKU-002": 19.99}

# ❌ BAD — trust whatever the client sends
def bad_checkout(cart):
    total = sum(item["price"] * item["qty"] for item in cart)
    return total

# ✅ BEST — look up the price server-side, validate types/ranges
@dataclass
class CartItem:
    sku: str
    qty: int

def best_checkout(raw_cart):
    items = []
    for raw in raw_cart:
        sku = str(raw.get("sku", ""))
        qty = int(raw.get("qty", 0))
        if sku not in PRICE_BOOK:
            raise ValueError(f"unknown SKU: {sku}")
        if not (1 <= qty <= 100):
            raise ValueError(f"bad qty: {qty}")
        items.append(CartItem(sku, qty))
    return sum(PRICE_BOOK[i.sku] * i.qty for i in items)

attacker_cart = [{"sku": "SKU-001", "qty": 1, "price": 0.01}]  # tampered price
print("BAD  total:", bad_checkout(attacker_cart), " ← attacker wins")
print("BEST total:", best_checkout(attacker_cart), " ← price came from our DB")

print("\nProduction tip: use pydantic or Django/Rails form validators.")
print("Azure WAF blocks obvious payloads, but business-logic validation belongs in your code.")


In [ ]:
# -------------------------------------------------------------------
# 5) Secret handling — Information Disclosure via repo / config
# -------------------------------------------------------------------
import os

# ❌ BAD — secret hardcoded in source, ends up in git history and container images
DB_PASSWORD_BAD = "P@ssw0rd!prod-2024"  # never do this

# ✅ BEST — secret pulled from the environment, injected by the platform
#     In Azure this comes from a Key Vault reference in App Service / Container Apps:
#       @Microsoft.KeyVault(SecretUri=https://kv-prod.vault.azure.net/secrets/db-password/)
#     The app just reads os.environ["DB_PASSWORD"] — never sees the Key Vault.
db_password = os.environ.get("DB_PASSWORD", "<not set — fetched from Key Vault at runtime>")
print("DB_PASSWORD:", db_password)

print("""
Principles:
  • Managed identity > client secret > hardcoded secret
  • Secrets never in code, config files, env-in-compose, or logs
  • Rotate automatically (Key Vault auto-rotation for supported services)
  • Detect leaks: GitHub secret scanning + push protection (Advanced Security)
""")


In [ ]:
# ===================================================================
# APPLICATION SECURITY QUIZ
# ===================================================================

QUIZ = [
    {
        'question': 'During threat modeling of an API, you identify that an attacker could modify\n'
                    'the price field in an HTTP request body. Which STRIDE category is this?',
        'options': {
            'A': 'Spoofing',
            'B': 'Tampering',
            'C': 'Information Disclosure',
            'D': 'Elevation of Privilege',
        },
        'answer': 'B',
        'explanation': 'Tampering = modifying data without authorization. Changing a price in the request body '
                       'is data modification (integrity violation). The mitigation is server-side validation — '
                       'never trust client-provided prices, always look up from the database.',
    },
    {
        'question': 'A GitHub Actions workflow needs to deploy resources to Azure.\n'
                    'What is the MOST secure authentication method?',
        'options': {
            'A': 'Store a service principal client secret in GitHub Secrets',
            'B': 'Use workload identity federation (OIDC) with a managed identity',
            'C': 'Store an Azure CLI refresh token in GitHub Secrets',
            'D': 'Use a personal access token (PAT) for Azure DevOps',
        },
        'answer': 'B',
        'explanation': 'Workload identity federation uses OIDC to exchange the GitHub-issued JWT token for an '
                       'Microsoft Entra ID token — no secrets stored in GitHub at all. The trust relationship is '
                       'a federated credential on the app registration or user-assigned managed identity. Client secrets leak and must '
                       'be rotated. PATs are user-bound and overly permissive.',
    },
    {
        'question': 'Tailspin Toys has a public-facing REST API and a web application.\n'
                    'Where should WAF be deployed for BEST protection?',
        'options': {
            'A': 'Azure Firewall in the hub VNet',
            'B': 'Azure Front Door with WAF policy',
            'C': 'NSG rules on the AKS subnet',
            'D': 'APIM rate limiting policies',
        },
        'answer': 'B',
        'explanation': 'Azure Front Door + WAF provides L7 protection at the edge — closest to the attacker. '
                       'It blocks OWASP Top 10 attacks (SQLi, XSS), provides bot protection, and handles DDoS '
                       'before traffic reaches your infrastructure. Azure Firewall is L4 (not web-aware). '
                       'NSGs are L3/L4. APIM rate limiting is complementary but not a WAF.',
    },
]

print('=== Application Security Architecture Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()